# 9. TeleopXR + Franka + MuJoCo（采集版）

本 notebook 目标：
1. 在 MuJoCo 中使用 Franka Panda 模型。
2. IK 使用 TeleopXR 库（FrankaRobot + PyrokiSolver + IKController）。
3. 食指 Trigger 控制夹爪开合。
4. 支持 RECORD=True/False 两种模式。


In [ ]:
from pathlib import Path
import os
import sys
import env_config

PROJECT_ROOT = Path.cwd()
TELEOPXR_ROOT = Path(os.environ.get("TELEOPXR_ROOT", "/path/to/teleop_xr")).expanduser()

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TELEOPXR_ROOT =', TELEOPXR_ROOT)

assert (PROJECT_ROOT / 'mujoco_env').exists(), 'Open this notebook from lerobot-mujoco-tutorial root.'
assert TELEOPXR_ROOT.exists(), f'TeleopXR source not found: {TELEOPXR_ROOT}. Set TELEOPXR_ROOT first.'

# Put local teleop_xr source path first in sys.path
if str(TELEOPXR_ROOT) not in sys.path:
    sys.path.insert(0, str(TELEOPXR_ROOT))


## 1) 安装与依赖

如果环境已安装，可跳过。


In [2]:
# %pip install -U pip setuptools wheel
# %pip install -e "{TELEOPXR_ROOT}"
# %pip install robot_descriptions


## 2) 初始化 TeleopXR IK 与 MuJoCo Franka

规则：双手 Grip/Squeeze 同时按住才进入控制，双击触发重置。


In [3]:
import time
import threading
import numpy as np
import mujoco
import mujoco.viewer
import xml.etree.ElementTree as ET
from pathlib import Path

from teleop_xr import Teleop
from teleop_xr.config import TeleopSettings
from teleop_xr.messages import XRState, XRDeviceRole, XRHandedness

from teleop_xr.ik.robots.franka import FrankaRobot
from teleop_xr.ik.solver import PyrokiSolver
from teleop_xr.ik.controller import IKController

from robot_descriptions import panda_mj_description

# ---------- TeleopXR server ----------
TELEOP_HOST = '0.0.0.0'
TELEOP_PORT = 4444

# ---------- Runtime ----------
CTRL_HZ = 60
RUN_SECONDS = 300

state_lock = threading.Lock()
shared = {
    'latest_state': None,   # XRState
    'reset_requested': False,
}

def on_xr_state(_pose, info_dict):
    try:
        state = XRState.model_validate(info_dict)
    except Exception:
        return
    with state_lock:
        shared['latest_state'] = state

teleop = Teleop(TeleopSettings(host=TELEOP_HOST, port=TELEOP_PORT, input_mode='controller'))
teleop.subscribe(on_xr_state)

server_thread = threading.Thread(target=teleop.run, daemon=True)
server_thread.start()

# ---------- IK ----------
robot = FrankaRobot()
solver = PyrokiSolver(robot)
controller = IKController(robot, solver)
q_target = np.array(robot.get_default_config(), dtype=np.float32)


def _build_franka_pnp_scene_xml():
    """Create a full scene xml: Franka + table + mug + plate + 3 cameras."""
    panda_xml = Path(panda_mj_description.MJCF_PATH)
    tree = ET.parse(panda_xml)
    root = tree.getroot()

    # Remove original keyframes because we add extra free joints (qpos size changes).
    keyframe = root.find('keyframe')
    if keyframe is not None:
        root.remove(keyframe)

    compiler = root.find('compiler')
    if compiler is None:
        compiler = ET.SubElement(root, 'compiler')
    compiler.set('angle', 'radian')
    compiler.set('meshdir', str((panda_xml.parent / 'assets').as_posix()))
    compiler.set('autolimits', 'true')

    option = root.find('option')
    if option is None:
        option = ET.SubElement(root, 'option')
    option.set('integrator', 'implicitfast')

    worldbody = root.find('worldbody')
    if worldbody is None:
        worldbody = ET.SubElement(root, 'worldbody')

    # Table and cameras (same semantics as collect_data)
    table_body = ET.SubElement(worldbody, 'body', name='front_object_table', pos='0 0 0')
    ET.SubElement(table_body, 'geom', name='front_object_table', type='box', size='1.0 0.7 0.4', pos='0.5 0 0.4', rgba='0.63 0.55 0.45 1')

    cam1 = ET.SubElement(table_body, 'body', name='camera_agent', pos='0.8 0.0 1.2')
    ET.SubElement(cam1, 'camera', name='agentview', pos='0 0 0', xyaxes='0 1 0 -0.5 0 0.707', fovy='60')

    cam2 = ET.SubElement(table_body, 'body', name='camera_top', pos='0.5 0.0 1.8')
    ET.SubElement(cam2, 'camera', name='topview', pos='0 0 0', xyaxes='0 -1 0 1 0 0', fovy='90')

    cam3 = ET.SubElement(table_body, 'body', name='camera_side', pos='0.3 -0.7 1.3')
    ET.SubElement(cam3, 'camera', name='sideview', pos='0 0 0', xyaxes='1 0 0 0 0 1', fovy='90')

    # Mug and plate (dynamic free bodies)
    mug = ET.SubElement(worldbody, 'body', name='body_obj_mug_5', pos='0.42 -0.10 0.86')
    ET.SubElement(mug, 'freejoint', name='mug_free')
    ET.SubElement(mug, 'geom', name='mug_geom', type='cylinder', size='0.035 0.05', rgba='0.85 0.15 0.15 1', density='250', friction='1.0 0.2 0.05')

    plate = ET.SubElement(worldbody, 'body', name='body_obj_plate_11', pos='0.56 0.08 0.82')
    ET.SubElement(plate, 'freejoint', name='plate_free')
    ET.SubElement(plate, 'geom', name='plate_geom', type='cylinder', size='0.09 0.012', rgba='0.95 0.95 0.95 1', density='300', friction='1.0 0.2 0.05')

    # Wrist/egocentric camera mounted to hand
    hand = None
    for b in worldbody.findall('.//body'):
        if b.get('name') in ('hand', 'panda_hand', 'tcp_link'):
            hand = b
            if b.get('name') == 'hand':
                break
    if hand is not None and not any((c.get('name') == 'egocentric') for c in hand.findall('camera')):
        ET.SubElement(hand, 'camera', name='egocentric', pos='0.0 -0.06 0.05', xyaxes='-1 0 0 0 0 1', fovy='90')

    out_dir = Path('./asset/generated')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_xml = out_dir / 'franka_pnp_scene.xml'
    tree.write(out_xml, encoding='utf-8', xml_declaration=False)
    return out_xml


scene_xml = _build_franka_pnp_scene_xml()
model = mujoco.MjModel.from_xml_path(str(scene_xml))
data = mujoco.MjData(model)

print('TeleopXR URL:', f'https://<LAN_IP>:{TELEOP_PORT}')
print('MuJoCo scene loaded from:', scene_xml)
print('IK actuated joints:', robot.actuated_joint_names)


{"host":"0.0.0.0","port":4444,"robot_vis":null,"input_mode":"controller","speed":1.0,"natural_phone_orientation_euler":[0.0,-0.7853981633974483,0.0],"natural_phone_position":[0.0,0.0,0.0],"camera_views":{},"video_config":null}
Server started at 0.0.0.0:4444
The phone web app should be available at https://192.168.209.36:4444
2026-03-01 16:30:25.572 | INFO     | pyroki._robot_urdf_parser:_topologically_sort_joints:199 - Joints were not in topological order; they will be internally sorted.
2026-03-01 16:30:26.658 | INFO     | jaxls._py310._problem:analyze:121 - Building optimization problem with 4 terms and 1 variables: 4 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
2026-03-01 16:30:29.444 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 1 costs, 1 variables each: manipulability_residual
2026-03-01 16:30:29.704 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 1 costs, 1 variables each: rest_residual
2026-03-01 16:30:29.866 | INFO     | jaxls._py3

TeleopXR URL: https://<LAN_IP>:4444
MuJoCo model loaded from: %USERPROFILE%\.cache\robot_descriptions\mujoco_menagerie\franka_emika_panda\panda.xml
IK actuated joints: ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']


In [4]:
import math
import teleop_xr
from teleop_xr.messages import XRDeviceRole, XRHandedness

# A. 放宽 pose jump 判定（减少频繁 reset）
_orig_are_close = teleop_xr.are_close
def _are_close_relaxed(a, b=None, lin_tol=1e-9, ang_tol=1e-9):
    # 线位移 20cm、角度 80° 以内不判定为 jump
    return _orig_are_close(a, b, lin_tol=0.20, ang_tol=math.radians(80))
teleop_xr.are_close = _are_close_relaxed

# B. deadman 改为“仅右手 Squeeze(Grip)”
def _deadman_right_only(self, state):
    for d in state.devices:
        if d.role == XRDeviceRole.CONTROLLER and d.handedness == XRHandedness.RIGHT:
            if d.gamepad is None:
                return False
            return (len(d.gamepad.buttons) > 1 and bool(d.gamepad.buttons[1].pressed))
    return False

controller._check_deadman = _deadman_right_only.__get__(controller, type(controller))
print("patched: deadman=RIGHT squeeze only, pose jump threshold relaxed")


patched: deadman=RIGHT squeeze only, pose jump threshold relaxed


## 3) 关节映射与辅助函数


In [5]:
def mat_to_rpy(R):
    # XYZ extrinsic (roll-pitch-yaw)
    sy = np.sqrt(R[0, 0] * R[0, 0] + R[1, 0] * R[1, 0])
    singular = sy < 1e-6
    if not singular:
        roll = np.arctan2(R[2, 1], R[2, 2])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = np.arctan2(R[1, 0], R[0, 0])
    else:
        roll = np.arctan2(-R[1, 2], R[1, 1])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = 0.0
    return np.array([roll, pitch, yaw], dtype=np.float32)


def _get_controller_button(xr_state, hand: str, btn_idx: int):
    if xr_state is None:
        return None
    for d in xr_state.devices:
        if d.role != XRDeviceRole.CONTROLLER:
            continue
        if hand == 'right' and d.handedness != XRHandedness.RIGHT:
            continue
        if hand == 'left' and d.handedness != XRHandedness.LEFT:
            continue
        if d.gamepad is None or len(d.gamepad.buttons) <= btn_idx:
            return None
        return d.gamepad.buttons[btn_idx]
    return None


def get_trigger_value(xr_state):
    b = _get_controller_button(xr_state, 'right', 0)  # trigger
    return float(b.value) if b is not None else 0.0


def get_squeeze_pressed(xr_state, hand: str):
    b = _get_controller_button(xr_state, hand, 1)  # squeeze / grip
    return bool(b.pressed) if b is not None else False


# Bimanual double-squeeze detector: both hands must double-press squeeze to reset/save.
reset_state = {
    'prev_left': False,
    'prev_right': False,
    'left_times': [],
    'right_times': [],
    'double_window_s': 0.40,
    'sync_window_s': 0.80,
}

def update_bimanual_double_squeeze(xr_state, now_s):
    l = get_squeeze_pressed(xr_state, 'left')
    r = get_squeeze_pressed(xr_state, 'right')

    # rising edges
    if l and (not reset_state['prev_left']):
        reset_state['left_times'].append(now_s)
    if r and (not reset_state['prev_right']):
        reset_state['right_times'].append(now_s)

    reset_state['prev_left'] = l
    reset_state['prev_right'] = r

    # keep recent only
    dw = reset_state['double_window_s']
    reset_state['left_times'] = [t for t in reset_state['left_times'] if now_s - t <= 2.0]
    reset_state['right_times'] = [t for t in reset_state['right_times'] if now_s - t <= 2.0]

    left_double = len(reset_state['left_times']) >= 2 and (reset_state['left_times'][-1] - reset_state['left_times'][-2] <= dw)
    right_double = len(reset_state['right_times']) >= 2 and (reset_state['right_times'][-1] - reset_state['right_times'][-2] <= dw)

    if left_double and right_double:
        if abs(reset_state['left_times'][-1] - reset_state['right_times'][-1]) <= reset_state['sync_window_s']:
            reset_state['left_times'].clear()
            reset_state['right_times'].clear()
            return True
    return False


def map_joint_qpos_indices(model, target_joint_names):
    model_joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(model.njnt)]
    model_joint_set = set(model_joint_names)

    idx = []
    valid_names = []

    for name in target_joint_names:
        candidates = [name]

        if name.startswith('panda_'):
            candidates.append(name.replace('panda_', '', 1))
        else:
            candidates.append('panda_' + name)

        if 'joint' in name:
            suffix = name[name.find('joint'):]
            candidates.append(suffix)

        hit = None
        for c in candidates:
            if c in model_joint_set:
                hit = c
                break

        if hit is not None:
            jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, hit)
            idx.append(int(model.jnt_qposadr[jid]))
            valid_names.append(hit)

    return valid_names, idx


ik_arm_joint_names = list(robot.actuated_joint_names[:7])
arm_joint_names, arm_qpos_idx = map_joint_qpos_indices(model, ik_arm_joint_names)

gripper_candidates = ['panda_finger_joint1', 'panda_finger_joint2', 'finger_joint1', 'finger_joint2']
gripper_names, gripper_qpos_idx = map_joint_qpos_indices(model, gripper_candidates)

print('IK arm joints:', ik_arm_joint_names)
print('Mapped arm joints:', list(zip(arm_joint_names, arm_qpos_idx)))
print('Mapped gripper joints:', list(zip(gripper_names, gripper_qpos_idx)))
assert len(arm_qpos_idx) == 7, f'Arm joint mapping failed, got {len(arm_qpos_idx)}'


mug_free_jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'mug_free')
plate_free_jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'plate_free')
mug_qadr = int(model.jnt_qposadr[mug_free_jid]) if mug_free_jid >= 0 else -1
plate_qadr = int(model.jnt_qposadr[plate_free_jid]) if plate_free_jid >= 0 else -1


def randomize_objects(data, seed=None):
    rng = np.random.default_rng(seed)
    for _ in range(200):
        mug_xy = np.array([rng.uniform(0.35, 0.55), rng.uniform(-0.20, 0.20)], dtype=np.float32)
        plate_xy = np.array([rng.uniform(0.35, 0.60), rng.uniform(-0.20, 0.20)], dtype=np.float32)
        if np.linalg.norm(mug_xy - plate_xy) >= 0.18:
            break

    if mug_qadr >= 0:
        data.qpos[mug_qadr:mug_qadr+3] = np.array([mug_xy[0], mug_xy[1], 0.86], dtype=np.float32)
        data.qpos[mug_qadr+3:mug_qadr+7] = np.array([1, 0, 0, 0], dtype=np.float32)
    if plate_qadr >= 0:
        data.qpos[plate_qadr:plate_qadr+3] = np.array([plate_xy[0], plate_xy[1], 0.82], dtype=np.float32)
        data.qpos[plate_qadr+3:plate_qadr+7] = np.array([1, 0, 0, 0], dtype=np.float32)

    mujoco.mj_forward(model, data)
    return np.concatenate([
        np.array([mug_xy[0], mug_xy[1], 0.86], dtype=np.float32),
        np.array([plate_xy[0], plate_xy[1], 0.82], dtype=np.float32),
    ], dtype=np.float32)


def get_body_pos(data, body_name):
    bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
    if bid < 0:
        return None
    return data.xpos[bid].copy()


def check_success_like_collect(data):
    p_mug = get_body_pos(data, 'body_obj_mug_5')
    p_plate = get_body_pos(data, 'body_obj_plate_11')
    p_hand = get_body_pos(data, 'hand')
    if p_mug is None or p_plate is None or p_hand is None:
        return False

    near_xy = np.linalg.norm(p_mug[:2] - p_plate[:2]) < 0.10
    near_z = abs(p_mug[2] - p_plate[2]) < 0.10

    grip_sum = 0.0
    for qidx in gripper_qpos_idx:
        grip_sum += float(data.qpos[qidx])
    gripper_open = grip_sum > 0.03

    ee_up = p_hand[2] > 0.75
    return bool(near_xy and near_z and gripper_open and ee_up)


def render_cam(renderer, data, cam_name, width=256, height=256):
    renderer.update_scene(data, camera=cam_name)
    img = renderer.render()
    return img


def apply_targets_to_mujoco(data, q_arm, gripper_open_ratio):
    n = min(len(q_arm), len(arm_qpos_idx))
    for i in range(n):
        data.qpos[arm_qpos_idx[i]] = q_arm[i]

    for jname, qidx in zip(gripper_names, gripper_qpos_idx):
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jname)
        if jid >= 0:
            qmin, qmax = model.jnt_range[jid]
            val = qmin + float(gripper_open_ratio) * (qmax - qmin)
            data.qpos[qidx] = val


IK arm joints: ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
Mapped arm joints: [('joint1', 0), ('joint2', 1), ('joint3', 2), ('joint4', 3), ('joint5', 4), ('joint6', 5), ('joint7', 6)]
Mapped gripper joints: [('finger_joint1', 7), ('finger_joint2', 8), ('finger_joint1', 7), ('finger_joint2', 8)]


## 4) 采集参数（record 开关）


In [6]:
import os
from PIL import Image
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

RECORD = True  # True: record dataset, False: debug only

REPO_NAME = 'datawhale_eai_franka_pnp_xr'
ROOT = './demo_data_franka_pnp_xr'
ROBOT_TYPE = 'franka'
TASK_NAME = 'Put mug cup on the plate'
NUM_DEMO = 3

# For pnp data collection, physics mode is recommended.
SIM_MODE = 'physics'   # 'physics' or 'kinematic'
SIM_FPS = 60
DATASET_FPS = 20
CAPTURE_EVERY_N = max(1, SIM_FPS // DATASET_FPS)
FPS = SIM_FPS

dataset = None
record_flag = False
episode_id = 0
obj_init_pose = randomize_objects(data)

if RECORD:
    create_new = True
    if os.path.exists(ROOT):
        print(f'Directory {ROOT} already exists.')
        ans = input('Delete and recreate it (y/n) ')
        if ans.strip().lower() == 'y':
            import shutil
            shutil.rmtree(ROOT)
        else:
            create_new = False

    if create_new:
        dataset = LeRobotDataset.create(
            repo_id=REPO_NAME,
            root=ROOT,
            robot_type=ROBOT_TYPE,
            fps=DATASET_FPS,
            features={
                'observation.image': {
                    'dtype': 'image',
                    'shape': (256, 256, 3),
                    'names': ['height', 'width', 'channels'],
                },
                'observation.wrist_image': {
                    'dtype': 'image',
                    'shape': (256, 256, 3),
                    'names': ['height', 'width', 'channels'],
                },
                'observation.state': {
                    'dtype': 'float32',
                    'shape': (6,),
                    'names': ['state'],
                },
                'action': {
                    'dtype': 'float32',
                    'shape': (8,),  # 7 arm + 1 gripper
                    'names': ['action'],
                },
                'obj_init': {
                    'dtype': 'float32',
                    'shape': (6,),
                    'names': ['obj_init'],
                },
            },
            image_writer_threads=10,
            image_writer_processes=5,
        )
    else:
        dataset = LeRobotDataset(REPO_NAME, root=ROOT)

print(f'SIM_MODE={SIM_MODE}, SIM_FPS={SIM_FPS}, DATASET_FPS={DATASET_FPS}, CAPTURE_EVERY_N={CAPTURE_EVERY_N}')
print('Cameras for rendering: agentview + egocentric + sideview')
print('Dataset uses: observation.image=agentview, observation.wrist_image=egocentric')


Directory ./demo_data_franka_xr already exists.


Delete and recreate it (y/n)  y


SIM_MODE=kinematic, SIM_FPS=90, DATASET_FPS=30, CAPTURE_EVERY_N=3


## 5) 主循环（IK 跟随 + 可选采集）

- 右手 Trigger 控制夹爪。
- RECORD=True 时写入训练数据。


In [8]:
renderer = mujoco.Renderer(model, 256, 256)

obs_buffer = []
act_buffer = []

start_t = time.time()
step_count = 0

# initialize robot posture
q_target = np.array(robot.get_default_config(), dtype=np.float32)
apply_targets_to_mujoco(data, q_target[:7], gripper_open_ratio=1.0)
mujoco.mj_forward(model, data)

with mujoco.viewer.launch_passive(model, data) as viewer:
    while viewer.is_running():
        tic = time.time()

        now = time.time()
        if (not RECORD) and (now - start_t > RUN_SECONDS):
            print('Debug mode timeout reached.')
            break

        with state_lock:
            xr_state = shared['latest_state']

        # manual episode-end trigger: both hands double-press squeeze
        reset_requested = update_bimanual_double_squeeze(xr_state, now)

        if reset_requested:
            controller.reset()
            q_target = np.array(robot.get_default_config(), dtype=np.float32)
            apply_targets_to_mujoco(data, q_target[:7], gripper_open_ratio=1.0)

            if RECORD and record_flag and dataset is not None:
                dataset.save_episode()
                episode_id += 1
                print(f'Episode saved by manual bimanual double-squeeze. episode_id={episode_id}')
                record_flag = False

            obj_init_pose = randomize_objects(data)

            if RECORD and episode_id >= NUM_DEMO:
                print('Target number of demos reached.')
                break

        # IK step
        if xr_state is not None:
            q_target = np.array(controller.step(xr_state, q_target), dtype=np.float32)

        deadman_active = bool(controller.active)
        trig = get_trigger_value(xr_state)
        gripper_open_ratio = float(np.clip(1.0 - trig, 0.0, 1.0))

        apply_targets_to_mujoco(data, q_target[:7], gripper_open_ratio)

        if SIM_MODE == 'kinematic':
            data.qvel[:] = 0.0
            if hasattr(data, 'qacc'):
                data.qacc[:] = 0.0
            mujoco.mj_forward(model, data)
        else:
            mujoco.mj_step(model, data)

        # camera rendering
        agent_image = render_cam(renderer, data, 'agentview')
        wrist_image = render_cam(renderer, data, 'egocentric')
        side_image = render_cam(renderer, data, 'sideview')

        # ee state (x,y,z,r,p,y)
        ee_body_name = 'hand'
        bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, ee_body_name)
        if bid < 0:
            bid = model.nbody - 1
        p = data.xpos[bid].copy()
        R = data.xmat[bid].reshape(3, 3).copy()
        rpy = mat_to_rpy(R)
        ee_state = np.concatenate([p, rpy], dtype=np.float32)

        action_vec = np.concatenate([q_target[:7], np.array([1.0 - gripper_open_ratio], dtype=np.float32)], dtype=np.float32)

        if RECORD:
            if deadman_active and (not record_flag):
                record_flag = True
                print(f'Start recording episode {episode_id + 1}')

            if record_flag and dataset is not None and (step_count % CAPTURE_EVERY_N == 0):
                dataset.add_frame(
                    {
                        'observation.image': np.asarray(Image.fromarray(agent_image).resize((256, 256))),
                        'observation.wrist_image': np.asarray(Image.fromarray(wrist_image).resize((256, 256))),
                        'observation.state': ee_state,
                        'action': action_vec,
                        'obj_init': obj_init_pose.astype(np.float32),
                    },
                    task=TASK_NAME,
                )

            # auto success criterion (similar to collect_data)
            if record_flag and check_success_like_collect(data):
                dataset.save_episode()
                episode_id += 1
                record_flag = False
                print(f'Episode saved by success criterion. episode_id={episode_id}')

                controller.reset()
                q_target = np.array(robot.get_default_config(), dtype=np.float32)
                apply_targets_to_mujoco(data, q_target[:7], gripper_open_ratio=1.0)
                obj_init_pose = randomize_objects(data)

                if episode_id >= NUM_DEMO:
                    print('Target number of demos reached.')
                    break
        else:
            obs_buffer.append(ee_state.copy())
            act_buffer.append(action_vec.copy())

        viewer.sync()

        step_count += 1
        if step_count % FPS == 0:
            l_sq = get_squeeze_pressed(xr_state, 'left')
            r_sq = get_squeeze_pressed(xr_state, 'right')
            print(
                f'steps={step_count}, mode={SIM_MODE}, deadman={deadman_active}, '
                f'trigger={trig:.2f}, gripper={1.0-gripper_open_ratio:.2f}, '
                f'left_sq={l_sq}, right_sq={r_sq}, record={record_flag}, episodes={episode_id}'
            )

        elapsed = time.time() - tic
        time.sleep(max(0.0, 1.0 / FPS - elapsed))


print('Loop finished.')
if RECORD:
    print(f'RECORD mode done. Episodes saved = {episode_id}')
else:
    print('DEBUG mode done.', 'frames =', len(obs_buffer), 'actions =', len(act_buffer))


Episode saved. episode_id=2
Target number of demos reached.
Loop finished.
RECORD mode done. Episodes saved = 2


## 6) 调试轨迹保存（仅 RECORD=False 生效）


In [ ]:
from pathlib import Path
import numpy as np

save_dir = Path('./demo_data_franka_xr_debug')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'franka_xr_trace.npz'

if not RECORD and len(obs_buffer) > 0:
    np.savez_compressed(
        save_path,
        obs=np.asarray(obs_buffer, dtype=np.float32),
        action=np.asarray(act_buffer, dtype=np.float32),
    )
    print('saved debug trace to', save_path.resolve())
else:
    print('No debug trace to save (or RECORD=True).')

if RECORD:
    print('Dataset root =', Path(ROOT).resolve())
    print('Repo name =', REPO_NAME)


## 7) 常见问题

1. 端口被占用：换端口或先 kill 进程。
2. 没有 XR 数据：确认头显网络与防火墙。
3. 夹爪不动：检查 trigger 变化和 gripper 映射。
